In [ ]:
# 变量表调试 Notebook

逐步调试 `create_tag_table_with_tags` 接口，排查"变量地址正常但名称/表名为空"的问题。

**调试流程：**
1. 加载 TIA API
2. 启动 TIA Portal 并创建项目
3. 添加 PLC 设备
4. 构造测试变量并调用 `create_tag_table_with_tags`
5. 回读变量表内容，验证写入结果
6. 清理会话

## 第 1 步：环境准备 —— 添加工程路径并加载 TIA API

In [7]:
# 若你刚修改过 openness/tia_core.py，这一格可强制重载模块
import importlib
import tia_core as tia_core

# 重置 API 已加载标记，并重载模块
try:
    tia_core._api_loaded = False
except Exception:
    pass
importlib.reload(tia_core)

print("✅ tia_core 已重载，后续将使用最新的 DLL 加载逻辑")

import sys
import os

# 将项目根目录加入 Python 路径，使 openness 包可导入
PROJECT_ROOT = r"E:\PlcProject\Code\PLC\SCDW"
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

API_DIR = r"E:\PlcProject\SoftWares\Siemens\Automation\Portal V17\PublicAPI\V17"

from openness.tia_core import load_tia_api, set_default_api_dir
set_default_api_dir(API_DIR)
load_tia_api(API_DIR)

print("✅ TIA API 加载完成")

✅ tia_core 已重载，后续将使用最新的 DLL 加载逻辑
✅ TIA API 加载完成


## 第 2 步：启动 TIA Portal 并创建测试项目

In [8]:
import os
import sys
import traceback
import clr  # type: ignore

from tia_core import load_tia_api, set_default_api_dir
from tia_core import start_tia_portal, create_project

PROJECT_ROOT_DIR = r"E:\PlcProject\Projects"
PROJECT_NAME = "TagDebugTest"

# 参考 init_tia_project：先 set_default_api_dir，再 load_tia_api，再启动/建项目
# 不修改 tia_core 启动代码，只在此单元补齐 Contract 依赖搜索路径
try:
    set_default_api_dir(API_DIR)

    tia_root = r"E:\PlcProject\SoftWares\Siemens\Automation\Portal V17"
    bin_dir = os.path.join(tia_root, "Bin")
    bin_public_api_dir = os.path.join(bin_dir, "PublicAPI")

    probe_dirs = [API_DIR, os.path.dirname(API_DIR), tia_root, bin_dir, bin_public_api_dir]

    for p in probe_dirs:
        if p and os.path.isdir(p) and p not in sys.path:
            sys.path.insert(0, p)

    env_path = os.environ.get("PATH", "")
    for p in probe_dirs:
        if p and os.path.isdir(p) and p not in env_path:
            os.environ["PATH"] = p + os.pathsep + os.environ.get("PATH", "")
            env_path = os.environ["PATH"]

    # 显式预加载 Contract，避免在导入 Siemens.Engineering 类型时才失败
    contract_dll = os.path.join(bin_public_api_dir, "Siemens.Engineering.Contract.dll")
    if os.path.isfile(contract_dll):
        clr.AddReference(contract_dll)

    load_tia_api(API_DIR)

    tia = start_tia_portal(with_ui=True)
    project = create_project(tia, PROJECT_ROOT_DIR, PROJECT_NAME, overwrite=True)

    print("✅ TIA 项目已创建")
    print(f"  项目名称：{PROJECT_NAME}")
    print(f"  项目路径：{os.path.join(PROJECT_ROOT_DIR, PROJECT_NAME)}")
    print("  TIA UI：有界面")
except Exception as exc:
    print(f"❌ 创建项目失败：{exc}")
    print(traceback.format_exc())

✅ TIA 项目已创建
  项目名称：TagDebugTest
  项目路径：E:\PlcProject\Projects\TagDebugTest
  TIA UI：有界面


## 第 3 步：添加 PLC 设备（CPU 1214C DC/DC/DC）

In [9]:
from openness.tia_hardware import add_plc_device

CPU_ORDER_NUMBER = "OrderNumber:6ES7 214-1BG40-0XB0/V4.5"
DEVICE_NAME      = "PLC_1"

device, plc_sw = add_plc_device(project, CPU_ORDER_NUMBER, DEVICE_NAME, DEVICE_NAME)
print(f"✅ PLC 设备已添加：{DEVICE_NAME}  订货号：{CPU_ORDER_NUMBER}")

✅ PLC 设备已添加：PLC_1  订货号：OrderNumber:6ES7 214-1BG40-0XB0/V4.5


In [ ]:
# 测试添加LAD梯形图代码块
